# KUMUTEVA vs kubectl-mtb

**The paper prints one thing from this notebook: the feature table in the next
cell.** Everything below it is the apparatus that computes that table's cells
and stands behind them if a reviewer asks — the per-benchmark correspondence,
the agreement counts, the disagreements. None of it goes in the paper. It
cannot: it would need kubectl-mtb's profile levels introduced, its nineteen
benchmarks mapped onto our properties, and a definition of agreement defended
before a single number meant anything.

The question is not which tool scores better. It is what each one can express,
and whether the answers it gives are about the tenancy boundary at all.

Run `experiments/run-mtb.sh` first, then copy its output here:

```sh
rsync -a ~/kumuteva-mtb/ new_fairness_results/mtb/
```

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
import utils.mtb as mtb
import build_comparison_table as table

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Where run-mtb.sh output is expected after the rsync above. Point it at
# ../kumuteva-mtb to read the runs in place without copying.
MTB_RESULTS_DIR = "./mtb"

catalogue = mtb.load_catalogue()
print(f"{catalogue['benchmark_count']} benchmarks, pinned at {catalogue['source_commit'][:12]}")


## The table

Generated, not written. A symbol grades **the tool's ability to assess**, not
the security of any solution: `OK` means the tool produces a valid verdict.

- `X` — the tool has no check for this capability.
- `X*` — it has none, yet still reports on it. The scorecard exists and does
  not describe the tenancy boundary. Worse than a gap: a gap is visible.
- `--` — the recorded runs cannot answer yet. Runs made before the targeting
  checks existed cannot be graded, and are never assumed to have passed them.

In [ ]:
blocks, facts = table.build(MTB_RESULTS_DIR)
print(table.render_markdown(blocks))

### What each run actually aimed at

The evidence behind the control-plane rows, and the source of the numbers the
paper's accompanying paragraph quotes.

Read `identity known` and `benchmarked the tenant's own workload namespace`
together: a scorecard produced when either is **no** is not a measurement of
the tenancy boundary, however complete it looks. That is the vcluster case —
`setup` creates a namespace called `tenant1` in the host cluster too, but it
holds vcluster's syncer, not the tenant's workloads.

In [ ]:
from IPython.display import Markdown

Markdown(facts)

---

# Supporting apparatus

Everything from here down justifies the cells above. It is the artifact, not
the paper.

## About the tool being compared against

kubectl-mtb lives in an archived repository. Its last commit was December 2021,
and it ships no release binary. The version used here is pinned so the
comparison stays reproducible.

It has **19 benchmarks**. Each returns pass or fail.

## What each benchmark maps to

`mtb_mapping.yaml` says, for every benchmark, which KUMUTEVA properties cover
the same ground. It is written by hand and meant to be read and argued with.

Three columns matter:

- **dimension** — `isolation` if the benchmark checks the tenant is *stopped*
  from doing something, `autonomy` if it checks the tenant *can* do it, and
  `not_covered` if KUMUTEVA has no equivalent.
- **confidence** — `partial` means we test something close but not identical, so
  a mismatch may just be the mapping.
- **kumuteva_properties** — how many of our checks the benchmark corresponds to.

In [ ]:
coverage = mtb.coverage()
coverage.sort_values(["kumuteva_dimension", "benchmark"])

### The same ground, at different resolution

Counting properties can mislead here. Nearly all of ours are reachable from some
benchmark, but one benchmark — "block access to other tenant resources" —
accounts for most of them on its own: it asks a single yes/no question about all
namespaced resources at once, where we return a separate graded answer for each
resource and verb.

So the difference is not what is covered, it is how finely.

In [ ]:
import subprocess

# Generated from the binary rather than counted by hand, so it cannot drift.
_ = subprocess.run(
    ["../target/release/kumuteva", "verify", "/dev/null", "/dev/null",
     "--list-properties", "/tmp/kumuteva-properties.json"],
    check=True, capture_output=True,
)
with open("/tmp/kumuteva-properties.json") as handle:
    inventory = json.load(handle)

resolution = mtb.resolution(inventory)
print(f"kubectl-mtb verdicts:  {resolution['mtb_verdicts']}")
print(f"KUMUTEVA properties:   {resolution['kumuteva_properties']}")
widest = resolution["widest_benchmark"]
print(f"widest benchmark:      {widest[1]} covers {widest[0]} properties alone")
print()
print("Nothing in kubectl-mtb reaches these:")
for subsystem, resource, operation in resolution["properties_unreached"]:
    print(f"  {subsystem:14s} {resource:22s} {operation}")

## Results

Loads every solution that was run. A solution where kubectl-mtb could not be
applied at all still appears — that absence is a result.

In [ ]:
results, verify, manifests = mtb.load_run_tree(MTB_RESULTS_DIR)

for solution, manifest in sorted(manifests.items()):
    applicable = manifest.get("namespace_model_applicable", True)
    mark = "yes" if applicable else "NO"
    print(f"{solution:16s} benchmarked: {mark:3s}  verdicts: {len(results[solution]):2d}")
    if not applicable:
        print(f"                 {manifest.get('not_applicable_reason', '')}")

### Where kubectl-mtb cannot be used at all

kubectl-mtb tests a namespace inside one cluster, using an impersonated user.
Some solutions give each tenant its own API server instead, so there is no such
namespace to point it at.

That is not a low score. It is a kind of tenancy the benchmark set cannot
describe.

In [ ]:
comparison = mtb.compare(results, verify)
summary = mtb.summarise(comparison)

print(f"cases:            {summary['cases']}")
print(f"not comparable:   {summary['not_comparable']} ({summary['not_comparable_pct']:.0f}%)")
print(f"compared:         {summary['compared_exact']}")
print(f"  agree:          {summary['agree']}")
print(f"  partial:        {summary['partial']}")
print(f"  disagree:       {summary['disagree']}")
print(f"excluded (approximate mappings): {summary['partial_mappings_excluded']}")

### Do the two tools agree?

Rows are what KUMUTEVA found, columns are what kubectl-mtb found.

The diagonal is agreement. Off it:

- **Hard + fail** — we found it blocked, kubectl-mtb found a way through. Our
  check may be too weak. Worth investigating.
- **None + pass** — we found no isolation, kubectl-mtb passed it. It may have
  missed something, or our mapping is wrong.
- **Soft** — blocked, but in a way that reveals the shared environment.
  kubectl-mtb only has pass and fail, so it records a pass. This row is where
  the graded scale says something the binary one cannot.

How these are counted was fixed in `utils/mtb.py` before the first run, so the
definition of "agreement" could not be chosen to suit the numbers.

Benchmarks whose mapping is approximate are left out; a mismatch there says more
about the mapping than about either tool.

In [ ]:
mtb.contingency(comparison)

### Every case, one row each

In [ ]:
columns = ["benchmark", "title", "solution", "dimension", "confidence",
           "mtb", "kumuteva", "verdict"]
comparison[comparison.verdict != mtb.NOT_COMPARABLE][columns].sort_values(
    ["verdict", "solution", "benchmark"]
)

### Disagreements

Each of these needs an explanation before the paper reports it. A disagreement
is as likely to be our defect as theirs.

In [ ]:
disagreements = comparison[comparison.verdict == mtb.DISAGREE]
if disagreements.empty:
    print("none")
else:
    for row in disagreements.itertuples():
        print(f"{row.solution:16s} {row.benchmark:20s} {row.title}")
        print(f"    kubectl-mtb: {row.mtb}    KUMUTEVA: {row.kumuteva}")
        print()

### What kubectl-mtb checks and we do not

Recorded so the gaps are visible rather than implied. These are ours to close or
to acknowledge.

In [ ]:
for benchmark in mtb.benchmarks():
    if benchmark.covered:
        continue
    print(f"{benchmark.benchmark_id}  {benchmark.title}  [{benchmark.category}]")
    print(f"    {benchmark.rationale.splitlines()[0]}")
    print()